In [1]:
# 环境准备
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from IPython.display import Image, display
import os

load_dotenv()
API_KEY = os.getenv("SILICON_API_KEY")
model = init_chat_model(
    "Qwen/Qwen3-8B",
    model_provider="openai",
    base_url="https://api.siliconflow.cn/v1",
    api_key=API_KEY,
    temperature=0.0
)


def display_graph(app):
    # 使用 Graphviz 渲染（Colab 最稳定的方案）
    try:
        display(Image(app.get_graph(xray=True).draw_png()))
    except Exception as e:
        print(f"Graphviz 渲染失败: {e}")
        print("\n使用 Mermaid 文本方式显示:")
        print(app.get_graph(xray=True).draw_mermaid())

In [3]:
from langgraph.constants import START, END
from langgraph.graph import StateGraph
import time
from typing import TypedDict, Annotated
from operator import add
from langgraph.checkpoint.memory import MemorySaver


class ProcessingState(TypedDict):
    total_items: int
    processed_items: Annotated[int, add]
    current_step: str
    progress_pct: float


def step1(state: ProcessingState) -> dict:
    """数据加载"""
    time.sleep(1)
    return {
        "processed_items": 20,
        "current_step": "数据加载",
        "progress_pct": 20.0
    }


def step2(state: ProcessingState) -> dict:
    """数据处理"""
    time.sleep(1.5)
    return {
        "processed_items": 50,
        "current_step": "数据处理",
        "progress_pct": 70.0
    }


def step3(state: ProcessingState) -> dict:
    """保存结果"""
    time.sleep(0.5)
    return {
        "processed_items": 30,
        "current_step": "保存结果",
        "progress_pct": 100.0
    }


graph = StateGraph(ProcessingState)
graph.add_node("load", step1)
graph.add_node("process", step2)
graph.add_node("save", step3)

graph.add_edge(START, "load")
graph.add_edge("load", "process")
graph.add_edge("process", "save")
graph.add_edge("save", END)

# 带 checkpoint 的 streaming
checkpointer = MemorySaver()
app_with_checkpoint = graph.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "task-1"}}

print("\n=== Streaming with Checkpoint ===")
for chunk in app_with_checkpoint.stream(
        {"total_items": 100, "processed_items": 0},
        config,
        stream_mode="updates"
):
    print(f"节点更新: {chunk}")

# 查看最终 checkpoint
final_state = app_with_checkpoint.get_state(config)
print(f"\n最终状态: {final_state.values}")


=== Streaming with Checkpoint ===
节点更新: {'load': {'processed_items': 20, 'current_step': '数据加载', 'progress_pct': 20.0}}
节点更新: {'process': {'processed_items': 50, 'current_step': '数据处理', 'progress_pct': 70.0}}
节点更新: {'save': {'processed_items': 30, 'current_step': '保存结果', 'progress_pct': 100.0}}

最终状态: {'total_items': 100, 'processed_items': 100, 'current_step': '保存结果', 'progress_pct': 100.0}
